In [ ]:
import json, subprocess, sys, time, hashlib, shutil, zipfile, resource
from pathlib import Path
import torch
input_root=Path('/kaggle/input')
print('Mounted Kaggle inputs:',[p.name for p in input_root.iterdir()])
manifest_candidates=list(input_root.rglob('shared-html-reviewed-pages.jsonl'))
if not manifest_candidates: raise FileNotFoundError('shared-html-reviewed-pages.jsonl not mounted under /kaggle/input')
input_root=manifest_candidates[0].parent
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Kaggle GPU was not allocated')
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['bash','-lc','apt-get update -qq && apt-get install -y -qq tesseract-ocr tesseract-ocr-vie'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','docling==2.129.0','rapidocr==3.9.2','onnxruntime==1.22.0'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','vietocr==0.3.13'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','einops==0.2.0','gdown==4.4.0'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.3'],check=True)
from importlib.metadata import version
import transformers
print('Transformers pinned:', transformers.__version__)
granite_weight_files=list(Path('/kaggle/input').rglob('model-00001-of-00002.safetensors'))
if len(granite_weight_files)!=1: raise RuntimeError(f'Expected one mounted Granite model snapshot, got {granite_weight_files}')
granite_snapshot=granite_weight_files[0].parent
artifacts_root=Path('/kaggle/working/docling-artifacts'); artifacts_root.mkdir(parents=True,exist_ok=True)
granite_link=artifacts_root/'ibm-granite--granite-vision-4.1-4b'
if not granite_link.exists(): granite_link.symlink_to(granite_snapshot,target_is_directory=True)
from docling.models.utils.hf_model_download import download_hf_model
layout_model_dir=artifacts_root/'docling-project--docling-layout-heron-101'
if not (layout_model_dir/'config.json').is_file(): download_hf_model('docling-project/docling-layout-heron-101',revision='main',local_dir=layout_model_dir)
print('Granite snapshot mounted:', granite_snapshot)
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import LayoutObjectDetectionOptions, PdfPipelineOptions, GraniteVisionTableStructureOptions, TableFormerMode, TableStructureOptions
from docling.datamodel.stage_model_specs import ObjectDetectionModelSpec
from docling.datamodel.object_detection_engine_options import TransformersObjectDetectionEngineOptions
from docling.document_converter import DocumentConverter, ImageFormatOption
Path('/kaggle/working/docling_vietocr.py').write_text('"""Experimental Docling OCR model using RapidOCR detection and VietOCR recognition."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Iterable\nimport re\nimport subprocess\nimport tempfile\nfrom pathlib import Path\nfrom typing import ClassVar, Type\n\nimport cv2\nimport numpy as np\nfrom PIL import Image\nfrom pydantic import Field\n\nfrom docling.datamodel.accelerator_options import AcceleratorOptions\nfrom docling.datamodel.base_models import Page\nfrom docling.datamodel.document import ConversionResult\nfrom docling.datamodel.pipeline_options import OcrOptions\nfrom docling_core.types.doc import BoundingBox, CoordOrigin\nfrom docling_core.types.doc.page import TextCell\nfrom docling.models.base_ocr_model import BaseOcrModel\nfrom docling.models.factories import get_ocr_factory\nfrom docling.models.stages.ocr.tesseract_utils import (\n    parse_tesseract_orientation,\n    tesseract_box_to_bounding_rectangle,\n)\nfrom docling.utils.profiling import TimeRecorder\n\n\nclass VietOcrOptions(OcrOptions):\n    kind: ClassVar[str] = "vietocr"\n    lang: list[str] = Field(default_factory=list)\n\n\nclass VietOcrModel(BaseOcrModel):\n    """Keep Docling\'s page regions and geometry while replacing text recognition."""\n\n    def __init__(\n        self,\n        *,\n        enabled: bool,\n        artifacts_path: Path | None,\n        options: VietOcrOptions,\n        accelerator_options: AcceleratorOptions,\n    ):\n        super().__init__(\n            enabled=enabled,\n            artifacts_path=artifacts_path,\n            options=options,\n            accelerator_options=accelerator_options,\n        )\n        self.scale = options.scale\n        import torch\n        from rapidocr import RapidOCR\n        from vietocr.tool.config import Cfg\n        from vietocr.tool.predictor import Predictor\n\n        config = Cfg.load_config_from_name("vgg_seq2seq")\n        config["device"] = "cuda:0" if torch.cuda.is_available() else "cpu"\n        config["dataset"]["image_max_width"] = 512\n        config["predictor"]["beamsearch"] = False\n        self.recognizer = Predictor(config)\n        self.detector = RapidOCR()\n        self.orientation_diagnostics: list[dict] = []\n        self._orientation_counts: dict[str | None, int] = {}\n        self.current_reference_id: str | None = None\n\n    @staticmethod\n    def _detect_orientation(image: Image.Image) -> tuple[int, int | None, float | None]:\n        """Return correction angle and OSD evidence; uncertain crops stay unrotated."""\n        with tempfile.NamedTemporaryFile(suffix=".png") as image_file:\n            image.save(image_file.name)\n            try:\n                output = subprocess.run(\n                    ["tesseract", image_file.name, "stdout", "--psm", "0", "-l", "osd"],\n                    capture_output=True,\n                    check=True,\n                    text=True,\n                ).stdout\n            except (OSError, subprocess.CalledProcessError):\n                return 0, None, None\n        orientation = re.search(r"Orientation in degrees:\\s*(\\d+)", output)\n        confidence = re.search(r"Orientation confidence:\\s*([\\d.]+)", output)\n        orientation_degrees = int(orientation.group(1)) if orientation else None\n        correction = parse_tesseract_orientation(orientation.group(1)) if orientation else 0\n        confidence_value = float(confidence.group(1)) if confidence else None\n        return correction, orientation_degrees, confidence_value\n\n    @classmethod\n    def get_options_type(cls) -> Type[OcrOptions]:\n        return VietOcrOptions\n\n    def __call__(\n        self, conv_res: ConversionResult, page_batch: Iterable[Page]\n    ) -> Iterable[Page]:\n        if not self.enabled:\n            yield from page_batch\n            return\n\n        for page in page_batch:\n            if page._backend is None or not page._backend.is_valid():\n                yield page\n                continue\n\n            with TimeRecorder(conv_res, "ocr"):\n                ocr_rects = self.get_ocr_rects(page)\n                cells: list[TextCell] = []\n                for ocr_rect in ocr_rects:\n                    if ocr_rect.area() == 0:\n                        continue\n                    crop = page._backend.get_page_image(\n                        scale=self.scale, cropbox=ocr_rect\n                    )\n                    doc_orientation, osd_orientation, osd_confidence = self._detect_orientation(crop)\n                    region_index = self._orientation_counts.get(self.current_reference_id, 0) + 1\n                    self._orientation_counts[self.current_reference_id] = region_index\n                    self.orientation_diagnostics.append({\n                        "reference_id": self.current_reference_id,\n                        "ocr_region_index_1based": region_index,\n                        "bbox": [ocr_rect.l, ocr_rect.t, ocr_rect.r, ocr_rect.b],\n                        "crop_size": list(crop.size),\n                        "osd_orientation_degrees": osd_orientation,\n                        "correction_clockwise": doc_orientation,\n                        "osd_confidence": osd_confidence,\n                    })\n                    if doc_orientation:\n                        crop = crop.rotate(-doc_orientation, expand=True)\n                    image_size = crop.size\n                    image = cv2.cvtColor(np.asarray(crop), cv2.COLOR_RGB2BGR)\n                    detected = self.detector(\n                        image, use_det=True, use_cls=False, use_rec=False\n                    )\n                    boxes = getattr(detected, "boxes", None)\n                    if boxes is None:\n                        continue\n\n                    for index, quad in enumerate(boxes):\n                        quad = np.asarray(quad, dtype=np.float32)\n                        tl, tr, br, bl = quad\n                        width = max(\n                            int(np.linalg.norm(tr - tl)),\n                            int(np.linalg.norm(br - bl)), 1,\n                        )\n                        height = max(\n                            int(np.linalg.norm(bl - tl)),\n                            int(np.linalg.norm(br - tr)), 1,\n                        )\n                        target = np.array(\n                            [[0, 0], [width - 1, 0], [width - 1, height - 1], [0, height - 1]],\n                            dtype=np.float32,\n                        )\n                        line = cv2.warpPerspective(\n                            image,\n                            cv2.getPerspectiveTransform(quad, target),\n                            (width, height),\n                        )\n                        text = str(\n                            self.recognizer.predict(\n                                Image.fromarray(cv2.cvtColor(line, cv2.COLOR_BGR2RGB))\n                            )\n                            or ""\n                        ).strip()\n                        if not text:\n                            continue\n                        left, top = quad.min(axis=0)\n                        right, bottom = quad.max(axis=0)\n                        box = BoundingBox(\n                            l=float(left), t=float(top), r=float(right), b=float(bottom),\n                            coord_origin=CoordOrigin.TOPLEFT,\n                        )\n                        cells.append(\n                            TextCell(\n                                index=index,\n                                text=text,\n                                orig=text,\n                                confidence=1.0,\n                                from_ocr=True,\n                                rect=tesseract_box_to_bounding_rectangle(\n                                    box,\n                                    original_offset=ocr_rect,\n                                    scale=self.scale,\n                                    orientation=doc_orientation,\n                                    im_size=image_size,\n                                ),\n                            )\n                        )\n                self.post_process_cells(cells, page, conv_res)\n            yield page\n\n\ndef register_vietocr() -> None:\n    for allow_external_plugins in (False, True):\n        factory = get_ocr_factory(allow_external_plugins=allow_external_plugins)\n        if VietOcrOptions not in factory.classes:\n            factory.register(VietOcrModel, "document-pipeline-experiment", __name__)\n')
sys.path.insert(0,'/kaggle/working')
from docling_vietocr import VietOcrOptions, register_vietocr
register_vietocr()
manifest_path=input_root/'shared-html-reviewed-pages.jsonl'
manifest=[json.loads(line) for line in manifest_path.read_text().splitlines() if line.strip()]
output_root=Path('/kaggle/working/raw_pages'); output_root.mkdir(parents=True,exist_ok=True)
options=PdfPipelineOptions(artifacts_path=artifacts_root,accelerator_options=AcceleratorOptions(device=AcceleratorDevice.CUDA,num_threads=2),do_ocr=True,do_table_structure=True,layout_options=LayoutObjectDetectionOptions(model_spec=ObjectDetectionModelSpec(name='layout_heron_101',repo_id='docling-project/docling-layout-heron-101'),engine_options=TransformersObjectDetectionEngineOptions(score_threshold=0.5)),table_structure_options=GraniteVisionTableStructureOptions(),ocr_options=VietOcrOptions())
converter=DocumentConverter(format_options={InputFormat.IMAGE:ImageFormatOption(pipeline_options=options)})
converter.initialize_pipeline(InputFormat.IMAGE)
pipeline=next(iter(converter.initialized_pipelines.values()))
started=time.perf_counter(); pages=[]
for item in manifest:
    page_started=time.perf_counter(); page_id=item['reference_id']; image=input_root/item['image_file']
    try:
        converted=converter.convert(image)
        (output_root/f'{page_id}.json').write_text(json.dumps(converted.document.export_to_dict(),ensure_ascii=False)+'\n')
        pages.append({'reference_id':page_id,'status':str(converted.status),'seconds':round(time.perf_counter()-page_started,3)})
        print(page_id,converted.status,flush=True)
    except Exception as exc:
        pages.append({'reference_id':page_id,'status':'failed','error':repr(exc),'seconds':round(time.perf_counter()-page_started,3)})
        print(page_id,'FAILED',repr(exc),flush=True)
metadata={'dataset':'letrongnhan/docling-pipeline-pilot-5','run_purpose':'pipeline_smoke_test_only','not_comparable_to_full_benchmark':True,'input_manifest_sha256':hashlib.sha256(manifest_path.read_bytes()).hexdigest(),'input_pages':len(manifest),'device':torch.cuda.get_device_name(0),'cuda_available':True,'docling_version':version('docling'),'ocr':'RapidOCR 3.9.2 detection + VietOCR vgg_seq2seq greedy; Tesseract OSD per OCR crop','layout_model':'docling-project/docling-layout-heron-101','layout_score_threshold':0.5,'table_cell_matching':'not applicable; Granite Vision generates cell text','table_backend':'Granite Vision 4.1 4B (Docling built-in)','table_crop_rotation':'disabled','table_structure':'Docling TableFormer accurate + OCR cell matching','inference_wall_seconds':round(time.perf_counter()-started,3),'successful_pages':sum(p['status']!='failed' for p in pages),'failed_pages':sum(p['status']=='failed' for p in pages),'pages':pages}
Path('/kaggle/working/kaggle-inference-metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2)+'\n')
shutil.make_archive('/kaggle/working/raw_pages','zip',root_dir=output_root)
print(json.dumps({k:metadata[k] for k in ('device','input_pages','successful_pages','failed_pages','inference_wall_seconds')}),flush=True)
